In [ ]:
import polars as pl
import altair as alt
from pathlib import Path
import json


In [31]:
#Log files
root = Path('..')
parquet_log = root / 'data/jails-data/output/processing_logs.parquet'
logs_df = pl.read_parquet(parquet_log)

logs_df


pdf_name,page_number,page_id,status,error_stage,error_message
str,i64,str,str,str,str
"""Huff, Anthony 1-15""",1,"""Huff, Anthony 1-15_p1""","""failed""","""parse_image_dict""","""OCR/parsing error: OpenCV(4.11…"
"""Magana, Yvette 1-14""",1,"""Magana, Yvette 1-14_p1""","""success""",null,null
"""Webster, Tiffany 1-9""",1,"""Webster, Tiffany 1-9_p1""","""success""",null,null
"""Cross, Anton 1-27 (suicide att…",1,"""Cross, Anton 1-27 (suicide att…","""success""",null,null
"""Sampson, Kevin 1-12 (suicide a…",1,"""Sampson, Kevin 1-12 (suicide a…","""success""",null,null
…,…,…,…,…,…
"""UO - FOIA January 2023""",611,"""UO - FOIA January 2023_p611""","""success""",null,null
"""UO - FOIA January 2023""",612,"""UO - FOIA January 2023_p612""","""success""",null,null
"""UO - FOIA January 2023""",613,"""UO - FOIA January 2023_p613""","""failed""","""parse_image_dict""","""OCR/parsing error: OpenCV(4.11…"


In [32]:
root = Path('..')
parquet_path = root / 'data/jails-data/output/jails_pdfs.parquet'
df = pl.read_parquet(parquet_path)
df.head()

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Name,Facility Type,Phone Number,Address,Date,Time of Day,AM or PM,Table Contents,Injuries?,Resulting Death?,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected
str,str,str,str,str,str,str,str,str,str,str,str,list[str],str,str,str,f64,i64,bool
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Williamson Coun…","""County ""","""___ Telephone #: 618-998-2100 ""","""Address: 404 N. Van Buren Mari…","""Date of Occurrence: 01/14/2021…","""_ Time of Occurrence: 235 ""","""No filled box found""","[""Detainees Involved"", ""Name :"", … ""oe ee""]","""No""","""No ""","""Magana, Yvette 1-14_p1""",89.095406,283,true
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: McHenry County …","""County ""","""__ Telephone #: 815-338-4741 _…","""Address: 2200 N. Seminary Ave.…","""Date of Occurrence: 01/09/2021…","""“ wy wiew Time of Occurrence: …","""p.m. ""","[""Detainees, Jnvolved"", ""Date of Birth"", … ""———t.""]","""No""","""No ""","""Webster, Tiffany 1-9_p1""",87.506757,296,true
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Kane County She…","""County ""","""__ Telephone #: 630-232-6840 ""","""daress: 3/W/)) IL Koute 56 Sui…","""Date of Occurrence: 01/27/2021…","""_ Time of Occurrence: 204: ""","""No filled box found""","[""Detainees Involved"", ""Date of Birth"", … ""Murder 1*, Agg Battery""]","""No""","""No filled box found""","""Cross, Anton 1-27 (suicide att…",87.860068,293,true
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: /¥lATTHSOns = o…","""ant. Inc ""","""elephone #: _70$ -St3~ 3) 3c _…","""a LT LO CC CC LT tn PE oe Oe ~…","""Date of Occurrence: i/ (Z]2 ( …","""ime of Occurrence: _O4 Se phan…","""Error: less than two boxes fou…","[""WELIIIC CS WIVOlVEU"", ""-o--HHe"", … ""Ae bel pIvVAriie""]","""No""","""Error: less than two boxes fou…","""Sampson, Kevin 1-12 (suicide a…",78.514851,303,true
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Fax: (217) 558-4004 ‘acility N…","""- ""","""__ Telephone #: ( b1 16 OLY Ba…","""‘acility Name: SASL a! eae A C…","""aAqaaress. eet o~ SF Never Pat…","""EAE AOL EE Ve a —_—_ City Stat…","""Error creating ROI""","[""Detainees Involved"", ""- Name"", … ""_ (det""]","""No""","""iA ""","""Paradee, Travis 1-16_p1""",81.046667,300,true


In [30]:
df['Table Contents'][259]

""
str
"""Detainees Involved"""
"""Date of Birth"""
"""Date Confined"""
"""-Arresting Charge"""
"""Lara; Gloria"""
"""09/06/1987"""
"""02/ 17720 19."""


In [25]:
df.select(
    pl.col("Table Contents").value_counts(sort=True).alias("counts")
).unnest("counts")

Table Contents,count
list[str],u32
"[""Bad parse""]",458
[],90
"[""Detainees Involved"", ""Date of Birth"", … ""Arresting Charge| 05/24/199103/11/2023| Bradley, Deandre| Armed Robbery""]",24
"[""Detainees Involved"", ""Date of Birth"", … ""Aggravated Battery""]",16
"[""Whe MCS VOW"", ""Isaiah Jackson"", … ""7/30/2019""]",13
…,…
"[""Dato of Birth"", ""Date Confined"", … ""BATTERY - MAKE PI- IYSICAL CONTAC""]",1
"[""(1 Fighting among Detainees"", ""C1] Restraints Used"", … ""| TAZ CO HOLD""]",1
"[""WOLAHItes mivoliveu"", ""ae ae"", … ""oo spammers""]",1


In [26]:
most_common = (df["Table Contents"]
               .explode()
               .value_counts()
               .sort("count", descending=True))

print("Most common Table Contents values:")
most_common.head(10)

Most common Table Contents values:


Table Contents,count
str,u32
"""Date Confined""",8735
"""Date of Birth""",8706
"""Detainees Involved""",8128
"""Arresting Charge""",7576
"""wld Mls vowel""",6358
"""_ Arresting Charge""",772
"""Whe MCS VOW""",735
"""TW Wt,""",725
"""Allie Mills""",725


In [ ]:
#clean_data = []
#for record in raw_data:
#    clean_record = {}
#    for key, value in record.items():
#        if isinstance(value, (dict, list)):
#            clean_record[key] = str(value) 
#        else:
#            clean_record[key] = value
#    clean_data.append(clean_record)


DataFrame has 110 rows and 17 columns
Saved to ../data/jails-data/output/jails_pdfs.parquet


In [27]:
key_fields_s1 = [
    'Facility Name',  'Phone Number', 'Date', 'Address'
]
key_fields_s2 = [
    'Facility Type', 'Time of Day', 'AM or PM'
]
key_fields_s3 = [
    'Table Contents'
]

def get_stats_by_section(sec: int =1):
    
    if sec == 1:
        fields = key_fields_s1
    elif sec == 2:
        fields = key_fields_s2
    elif sec == 3:
        fields = key_fields_s3
    stats = []
    total = df.height

    for field in fields:
        print(field)
        if sec == 3:
           missing_count = df.filter(
               pl.col(field).is_null() |
               (pl.col(field).is_not_null() & (pl.col(field).list.len() == 0))).height
        else: 
        # Count nulls, "N/A", "None", and empty strings
            missing_count = df.filter(
                pl.col(field).is_null() | 
                pl.col(field).eq("N/A") | 
                pl.col(field).eq("None") |
                pl.col(field).eq("")
            ).height
        
        stats.append({
            "Field": field,
            "Missing_Count": missing_count,
            "Missing_Percent": (missing_count / total) * 100,
            "Complete_Count": total - missing_count,
            "Complete_Percent": ((total - missing_count) / total) * 100
        })
        stats_df = pl.DataFrame(stats)
    return stats_df

# Convert stats to DataFrame
stats1 = get_stats_by_section(1)  
stats2 = get_stats_by_section(2)  
stats3 = get_stats_by_section(3)  
stats_df = pl.concat([
    stats1.with_columns(pl.lit("Basic Info").alias("Section")),
    stats2.with_columns(pl.lit("Secondary Info").alias("Section")),
    stats3.with_columns(pl.lit("Table Data").alias("Section"))
])

stats_df.sort("Missing_Percent", descending=True)


Facility Name
Phone Number
Date
Address
Facility Type
Time of Day
AM or PM
Table Contents


Field,Missing_Count,Missing_Percent,Complete_Count,Complete_Percent,Section
str,i64,f64,i64,f64,str
"""Address""",1705,6.418219,24860,93.581781,"""Basic Info"""
"""Facility Name""",733,2.75927,25832,97.24073,"""Basic Info"""
"""Date""",579,2.17956,25986,97.82044,"""Basic Info"""
"""Time of Day""",284,1.069076,26281,98.930924,"""Secondary Info"""
"""Facility Type""",280,1.054018,26285,98.945982,"""Secondary Info"""
"""Phone Number""",212,0.798043,26353,99.201957,"""Basic Info"""
"""AM or PM""",134,0.504423,26431,99.495577,"""Secondary Info"""
"""Table Contents""",90,0.338792,26475,99.661208,"""Table Data"""


In [28]:
missing_chart = alt.Chart(stats_df).mark_bar().encode(
    x=alt.X('Missing_Percent:Q', title='% Missing'),
    y=alt.Y('Field:N', sort='-x', title='Field'),
    color=alt.value('#e45756'),  # Red color for missing
    tooltip=['Field', 'Missing_Count', alt.Tooltip('Missing_Percent:Q', format='.1f')]
).properties(
    width=600,
    title='Percentage of Missing Values by Field'
)

missing_chart

alt.Chart(...)

In [29]:
# Create a stacked bar chart showing complete vs missing
stacked_df = stats_df.unpivot(
    index=["Field"], 
    on=["Complete_Percent", "Missing_Percent"],
    variable_name="Status", 
    value_name="Percentage"
)

# Clean up status names
stacked_df = stacked_df.with_columns(
    pl.col("Status").map_elements(lambda x: "Complete" if "Complete" in x else "Missing")
)

# Create stacked bar chart
stacked_chart = alt.Chart(stacked_df).mark_bar().encode(
    x=alt.X('Percentage:Q', title='Percentage (%)'),
    y=alt.Y('Field:N', title='Field'),
    color=alt.Color('Status:N', scale=alt.Scale(
        domain=['Complete', 'Missing'],
        range=["#94adc7", "#e87e7e"]
    )),
    tooltip=['Field', 'Status', alt.Tooltip('Percentage:Q', format='.1f')]
).properties(
    width=600,
    title='Completion Rate by Field'
)

stacked_chart

/var/folders/m8/p0y07mnx0xj8nxd8l9zj7fn00000gn/T/ipykernel_2609/2304937129.py:10: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  stacked_df = stacked_df.with_columns(


alt.Chart(...)

In [ ]:
### TODO: Which pdf has the most missing all over the sections? prioritize


In [56]:
raw_data[15]

{'Deceased Cause, Date, and Time': 'N/A',
 'Deceased on Suicide Watch': 'N/A',
 'Deceased Reporter': 'N/A',
 'Deceased Examined by Physician': 'N/A',
 'Deceased Signs of Illness': 'N/A',
 'Facility Name': '‘acility Name: Cook County Department Of Corrections\noe JVINN GC CKalifarnia Nivicinn 117 Chica,\n',
 'Facility Type': 'None',
 'Phone Number': 'Telephone #: 773.674.3474\n~ Wl CNOoONno\n',
 'Address': '\\ddress: 2700 S. California Division 11 Chicago IL 60608\n',
 'Date': 'Date of Occurrence: 3/5/2021\n',
 'Time of Day': 'Time of Occurrence: 9:15:00 AM\n',
 'AM or PM': 'None',
 'Occurrence Dictionary': {'(1452, 267)': ['Other fenarijfy\\*\n', 0.0],
  '(1450, 192)': ['Assault among Vetainees\n', 0.09375],
  '(919, 186)': ['ASSé\n', 0.11753902662993572],
  '(449, 183)': ['ex\n', 0.09185606060606061],
  '(1707, 104)': ['Seric\n', 0.09090909090909091],
  '(453, 104)': ['CsoGd\n', 0.0634765625],
  '(1446, 103)': ['rire\n', 0.09090909090909091],
  '(916, 103)': ['hovc\n', 0.1051136363636

In [19]:
df.head(5)

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Type,Facility Name,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence Dictionary,Table Contents,Injuries?,Resulting Death?,Report ID
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""Facility Name: Qian ‘ Lira ar …","""e veto TI O71~- TELE: ""","""( Address: 24 / Spree LEFF P I…","""Date of Occurrence: / ~7 on, 7…","""* 4 WNT Vw _ Time of Occurrenc…","""None""","""{'(1414, 262)': ['Other (speci…","""['Detainees Involved', 'Ree.',…","""No""","""None""","""handwritten_test_p1"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""‘ Ballity Name: Fu\eia Cosuini…","""Telephone #: 309. $Y F- ah of …","""dress; OUD Vb Kec ded nel) 1 T…","""Nee Y Dats of Occurrence: Zrfo…","""Td ated fe “ive Time of Occurr…","""None""","""{'(27, 229)': ['Figniing\n', 0…","""['-— ge re ee ee eee ee', '—',…","""No""","""None""","""Part 6_p1"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""— 0 Hy . a ERE a Sirnal Nw ""","""1! Shou 12 G/ 58 ""","""Jate of Oecurrenceal AID 4 \ T…","""i : eee ee Type of Occurranca:…","""la el tee dt tie 7 eae | athe.…","""None""","""{'(1036, 137)': ['Oc Spr\n', 0…","""['at', '=', '2.', '_—', '=', '…","""No""","""None""","""Part 6_p2"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Cot ""","""-acility Name: DuPage County J…","""____ Telephone #: (630) 407-22…","""Address: 501 N. County Farm Rd…","""Date of Occurrence: 03/02/2021…","""_ Time of Occurrence: 2000 ""","""pm. ""","""{'(24, 265)': ['Fightine\n', 0…","""['a', '—— we —__—', 'Detainees…","""No""","""None""","""Part 6_p3"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Cou ""","""Facility Name: St. Clair Count…","""__ Telephone #: 618-277-3505 ""","""Address: 700 North 5"" Street B…","""Date of Occurrence: 03/18/2021…","""_ Time of Occurrence: 1:27 ""","""a.m. ""","""{'(40, 262)': ['Fightinc\n', 0…","""['lJelainees invoiveci', 'a', …","""No""","""Yes ""","""Part 6_p4"""
